In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import pickle
if not os.path.exists("tmp"):
	os.mkdir("tmp")
if not os.path.exists("fig"):
	os.mkdir("fig")

from typing import cast
from datetime import datetime, timedelta, UTC
from tqdm.auto import tqdm

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed, Adjustment

import polars as pl
import pandas as pd
import numpy as np

import yfinance as yf

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
pd.options.plotting.backend = "plotly"


In [ ]:
YEAR = 252
MONTH = 21
WEEK = 5
HOURS_PER_DAY = 6.5

# Alpaca client (shared instance used by both fetch_vix and fetch_data)
APCA_CLIENT = StockHistoricalDataClient(
	api_key=os.environ.get("APCA_API_KEY_ID"),
	secret_key=os.environ.get("APCA_API_SECRET_KEY"),
)


In [ ]:
# VIX DATA FETCH
# Downloads VIX (30-day), VIX9D (9-day), VIX3M (93-day) from Yahoo Finance.
# These are the CBOE index symbols: ^VIX, ^VIX9D, ^VIX3M.

VIX_TICKERS = {"^VIX":"vix", "^VIX3M":"vix3m" , "^VIX9D":"vix9d"}

def fetch_vix(start_date: datetime, end_date: datetime) -> pl.DataFrame:
	"""
	Download daily VIX term-structure closing values from Yahoo Finance.
	Returns a Polars DataFrame with columns: date, vix, vix3m, vix9d.
	Values are in percentage-point form (e.g. 20.0 = 20% annualised vol).
	VIX9D is included for diagnostics; only VIX and VIX3M drive the strategy.
	"""
	raw = yf.download(
		list(VIX_TICKERS.keys()),
		start=start_date,
		end=end_date + timedelta(days=1),
		auto_adjust=False,
		progress=False,
	)["Close"].rename(columns=VIX_TICKERS) # type: ignore
	
	df = (
		pl.from_pandas(raw.reset_index())
		.rename({"Date": "date"})
		.with_columns(pl.col("date").cast(pl.Date))
		.sort("date")
		.with_columns(
			pl.col("vix").forward_fill(),
			pl.col("vix3m").forward_fill(),
			pl.col("vix9d").forward_fill(),
		)
		.drop_nulls(subset=["vix", "vix3m"])
	)
	return df


In [ ]:
# VOL REGIME
# Computes a causal vol regime signal directly from VIX term-structure data.
#
# Two ingredients:
#   vix_ema     : short EMA of VIX — smooths day-to-day noise while staying
#                 reactive to genuine vol spikes.  Controlled by ema_span.
#   vol_zscore  : rolling z-score of vix_ema vs a trailing window — makes the
#                 "is vol high?" question regime-relative rather than absolute.
#                 Controlled by zscore_window.
#
# These two levers are the only parameters.  Simpler = fewer ways to overfit.

def compute_vol_regime(
	df_vix: pl.DataFrame,
	ema_span:      int = 5,    # EMA smoothing span (bars).  5 ≈ 1 trading week.
	zscore_window: int = 63,   # Lookback for rolling mean/std (bars).  63 ≈ 1 quarter.
) -> pl.DataFrame:
	"""
	Produce daily vol regime signals from raw VIX term-structure data.

	All computations are strictly causal — no future information used.

	Columns added to df_vix:
		vix_ema       : EMA-smoothed VIX (ema_span bars)
		vol_zscore    : (vix_ema[t] - mean(vix_ema[t-zscore_window:t]))
						  / std(vix_ema[t-zscore_window:t])
						Uses only past data; NaN for first zscore_window bars.
		term_slope    : VIX3M - VIX  (positive = normal contango,
									   negative = backwardation / stress)
		term_slope_ma : 21-bar causal trailing MA of term_slope
		vol_d         : daily vol move fraction for band-width computation
						= vix_ema / 100 / sqrt(YEAR)
		stress_flag   : term_slope < 0
	"""
	n   = len(df_vix)
	vix = df_vix["vix"].to_numpy()

	# EMA — Polars ewm_mean is consistent with the MAs used in backtest()
	vix_ema = pl.Series(vix).ewm_mean(span=ema_span).to_numpy()

	# Causal rolling z-score: window ends at t-1 so today's value is out-of-sample
	vol_zscore = np.full(n, np.nan)
	for i in range(zscore_window, n):
		w     = vix_ema[i - zscore_window : i]   # strictly past; excludes bar i
		mu, sigma = w.mean(), w.std()
		vol_zscore[i] = (vix_ema[i] - mu) / sigma if sigma > 0 else 0.0

	vix3m      = df_vix["vix3m"].to_numpy()
	term_slope = vix3m - vix

	# 21-bar causal trailing MA of slope
	slope_ma = np.full(n, np.nan)
	for i in range(21, n):
		slope_ma[i] = term_slope[i - 21 : i].mean()

	return (
		df_vix
		.with_columns([
			pl.Series("vix_ema",       vix_ema),
			pl.Series("vol_zscore",    vol_zscore),
			pl.Series("term_slope",    term_slope),
			pl.Series("term_slope_ma", slope_ma),
			# vol_d: per-bar fraction of price the vol band spans
			pl.Series("vol_d", vix_ema / 100.0 * np.sqrt(1.0 / YEAR)),
		])
		.with_columns(
			stress_flag = pl.col("term_slope") < 0,
		)
		.drop_nulls()   # drops first zscore_window rows where vol_zscore is NaN
	)


In [ ]:
# FETCH DATA

def fetch_data(symbol: str, start_date: datetime, end_date: datetime,
			   timeframe: TimeFrame,
			   client: StockHistoricalDataClient = APCA_CLIENT):

	# ── 1. VIX term structure ────────────────────────────────────────────────
	df_vix = fetch_vix(start_date, end_date)

	# ── 2. Intraday OHLCV bars (IEX feed, split + dividend adjusted) ─────────
	request = StockBarsRequest(
		symbol_or_symbols=symbol,
		timeframe=timeframe,
		start=start_date,
		end=end_date,
		feed=DataFeed.IEX,
		adjustment=Adjustment.ALL,
	)
	bars = client.get_stock_bars(request)
	df_ohlcv = pl.from_pandas(bars.df.loc[symbol].reset_index())  # type: ignore

	return df_ohlcv, df_vix


In [ ]:
# VIX DATA QUALITY CHECK

def validate_vix_alignment(df_vix: pl.DataFrame, df_ohlcv: pl.DataFrame,
							max_gap_days: int = 3):
	"""Check VIX date coverage vs OHLCV trading dates; flag long fill runs."""
	trading_dates = (
		df_ohlcv
		.with_columns(pl.col("timestamp").dt.date().alias("date"))
		["date"].unique().sort()
	)
	vix_dates = set(df_vix["date"].to_list())
	missing = [d for d in trading_dates.to_list() if d not in vix_dates]
	coverage = 1.0 - len(missing) / len(trading_dates)
	print(f"VIX date coverage: {coverage:.1%}  ({len(missing)} trading days missing)")
	if missing:
		print(f"  First 5 missing: {missing[:5]}")

	# Flag long forward-fill runs (consecutive identical VIX values)
	vix_arr        = df_vix["vix"].to_numpy()
	vix_dates_list = df_vix["date"].to_list()
	gaps, gap_start = [], None
	for i in range(1, len(vix_arr)):
		if vix_arr[i] == vix_arr[i - 1]:
			if gap_start is None:
				gap_start = i - 1
		else:
			if gap_start is not None:
				gap_len = i - gap_start
				if gap_len >= max_gap_days:
					gaps.append((vix_dates_list[gap_start], vix_dates_list[i - 1], gap_len))
				gap_start = None

	if gaps:
		print(f"\n{len(gaps)} forward-fill runs >= {max_gap_days} days:")
		for start, end, length in gaps[:10]:
			print(f"   {start} -> {end}  ({length} days)")
	else:
		print(f"No stale forward-fill runs >= {max_gap_days} days detected.")
	return missing, gaps

In [ ]:
# MAKE STRATEGY

from math import ceil
from typing import NamedTuple, Literal

def bars_per_day(timeframe: TimeFrame):
	if timeframe.unit == TimeFrameUnit.Hour:
		hours = timeframe.amount
	elif timeframe.unit == TimeFrameUnit.Minute:
		hours = timeframe.amount / 60
	else:
		raise ValueError("unit must be 'H' or 'm'")
	return HOURS_PER_DAY / hours

class StrategyHyperparams(NamedTuple):
	timeframe: TimeFrame
	# ── Vol regime parameters ────────────────────────────────────────────────
	band_std_dev: float
	ema_span: int
	zscore_window: int
	vol_regime_threshold: float
	use_stress_flag: bool
	# ── VWAP anchor control ──────────────────────────────────────────────────
	anchor_recompute: float
	# ── Trend following parameters ───────────────────────────────────────────
	trend_ma_window: int
	# ── Risk parameters ──────────────────────────────────────────────────────
	atr_period: float
	atr_multiplier: float
	
	@property
	def bpd(self) -> float:
		return bars_per_day(self.timeframe)
	
	@property
	def trend_ma_window_bars(self) -> int:
		return ceil(self.trend_ma_window * self.bpd)
	
	@property
	def anchor_recompute_bars(self) -> int:
		return ceil(self.anchor_recompute * self.bpd)
	
	@property
	def atr_period_bars(self) -> int:
		return ceil(self.atr_period * self.bpd)

def backtest(
	df_ohlcv:      pl.DataFrame,
	df_vix:        pl.DataFrame,
	params:        StrategyHyperparams,
	mr_exposure:   float = 1.0,
	tf_exposure:   float = 1.0,
	commission:    float = 0.0001,
	slippage:      float = 0.0002,
	entry_lag:     int   = 1,
	exit_lag:      int   = 0,
	long_only:     bool  = False,
	strategy_type: Literal["combined", "mean_reversion", "trend"] = "combined",
 ) -> pl.DataFrame:

	if params.anchor_recompute_bars < 0:
		raise ValueError("anchor_recompute_bars must be >= 0")

	# ── Compute vol regime from raw VIX ──────────────────────────────────────
	df_regime = compute_vol_regime(df_vix, ema_span=params.ema_span, zscore_window=params.zscore_window)

	# ── Join regime to OHLCV + construct causal VWAP anchor ─────────────────
	block_expr = (
		(pl.col("session_bar_idx") // params.anchor_recompute_bars).cast(pl.Int64)
		if params.anchor_recompute_bars > 0
		else pl.lit(0).cast(pl.Int64)
	)

	# ── Prepare OHLCV
	df = (
		df_ohlcv
		.with_columns(date=pl.col("timestamp").dt.date())
		.join(df_regime, on="date", how="left")
		.sort("timestamp")
		.fill_null(strategy="forward")
		.with_columns(
			volume_clean    = pl.col("volume").fill_null(0.0),
			session_bar_idx = pl.col("close").cum_count().over("date") - 1,
		)
		.with_columns(block_id = block_expr)
		.with_columns(
			vwap_num_cum = (pl.col("close") * pl.col("volume_clean")).cum_sum().over("date"),
			vwap_den_cum = pl.col("volume_clean").cum_sum().over("date"),
			vwap_num_blk = (pl.col("close") * pl.col("volume_clean")).cum_sum().over(["date", "block_id"]),
			vwap_den_blk = pl.col("volume_clean").cum_sum().over(["date", "block_id"]),
		)
		.with_columns(
			vwap_anchor_cum = pl.when(pl.col("vwap_den_cum") > 0).then(pl.col("vwap_num_cum") / pl.col("vwap_den_cum")).otherwise(pl.col("close")),
			vwap_anchor_blk = pl.when(pl.col("vwap_den_blk") > 0).then(pl.col("vwap_num_blk") / pl.col("vwap_den_blk")).otherwise(pl.col("close")),
		)
		.with_columns(
			anchor = (
				pl.col("vwap_anchor_blk") if params.anchor_recompute_bars > 0 else pl.col("vwap_anchor_cum")
			).shift(1).fill_null(pl.col("close")),
		)
		.with_columns(
			is_high_vol = pl.col("vol_zscore") > params.vol_regime_threshold,
			vol_move_dist = pl.col("anchor") * pl.col("vol_d") * params.band_std_dev,
			atr = pl.max_horizontal([
				pl.col("high") - pl.col("low"),
				(pl.col("high") - pl.col("close").shift(1)).abs(),
				(pl.col("low")  - pl.col("close").shift(1)).abs(),
			]).rolling_mean(window_size=params.atr_period_bars),
		)
	)

	# ── Trend filter ─────────────────────────────────────────────────────
	df = df.with_columns(
		trend_ma = pl.col("close").ewm_mean(span=params.trend_ma_window_bars)
	).with_columns(
		trend_slope    = pl.col("trend_ma") - pl.col("trend_ma").shift(1),
		trend_strength = pl.col("close") - pl.col("trend_ma"),
	).with_columns(
		trend_up       = (pl.col("trend_slope") > 0) & (pl.col("trend_strength") > pl.col("atr") * 0.5),
		trend_down     = (pl.col("trend_slope") < 0) & (pl.col("trend_strength") < -pl.col("atr") * 0.5)
	)

	# ── Mean reversion entries ───────────────────────────────────────────
	df = df.with_columns(
		upper_band = pl.col("anchor") + pl.col("vol_move_dist"),
		lower_band = pl.col("anchor") - pl.col("vol_move_dist"),
	).with_columns(
		cross_upper = (pl.col("high") > pl.col("upper_band")).shift(1) & (pl.col("high") < pl.col("upper_band")),
		cross_lower = (pl.col("low")  < pl.col("lower_band")).shift(1) & (pl.col("low")  > pl.col("lower_band")),
	).with_columns(
		mr_long_entry = (
			pl.lit(strategy_type in {"combined", "mean_reversion"})
			& pl.col("is_high_vol")
			& (~pl.lit(params.use_stress_flag) | pl.col("stress_flag"))
			& pl.col("cross_lower")
		).shift(entry_lag),
		mr_short_entry = (
			pl.lit(strategy_type in {"combined", "mean_reversion"})
			& ~pl.lit(long_only)
			& pl.col("is_high_vol")
			& (~pl.lit(params.use_stress_flag) | pl.col("stress_flag"))
			& pl.col("cross_upper")
		).shift(entry_lag),
	)

	# ── Trend entries ────────────────────────────────────────────────────
	df = df.with_columns(
		tf_long_entry = (
			pl.lit(strategy_type in {"combined", "trend"})
			& ~pl.col("is_high_vol")
			& pl.col("trend_up") & ~pl.col("trend_up").shift(1).fill_null(False)
		).shift(entry_lag),
		tf_short_entry = (
			pl.lit(strategy_type in {"combined", "trend"})
			& ~pl.lit(long_only)
			& ~pl.col("is_high_vol")
			& pl.col("trend_down") & ~pl.col("trend_down").shift(1).fill_null(False)
		).shift(entry_lag),
	)
	
	# ── State machine ─────────────────────────────────────────────────────
	df = df.with_columns(
		state        = pl.lit(0.0).cast(pl.Float64),
		active_stop  = pl.lit(np.nan).cast(pl.Float64),
		exit_marker  = pl.lit(0.0).cast(pl.Float64),
		entry_marker = pl.lit(0.0).cast(pl.Float64),
	)

	curr_state = 0
	exit_countdown = -1
	hwm = 0.0

	for i in range(1, len(df)):
		curr_open        = cast(float, df[i, "open"])
		curr_high        = cast(float, df[i, "high"])
		curr_low         = cast(float, df[i, "low"])
		curr_is_high_vol = cast(bool,  df[i, "is_high_vol"])
		curr_trend_up    = cast(bool,  df[i, "trend_up"])
		curr_trend_down  = cast(bool,  df[i, "trend_down"])
		prev_atr         = cast(float, df[i-1, "atr"])

		if exit_countdown == 0:
			df[i, "exit_marker"] = curr_state
			curr_state = 0
			exit_countdown = -1
		elif exit_countdown > 0:
			exit_countdown -= 1

		if curr_state != 0 and exit_countdown == -1:
			should_exit = False
			if abs(curr_state) == 1:
				if curr_state > 0:
					stop_level = hwm - (prev_atr * params.atr_multiplier)
					df[i, "active_stop"] = stop_level
					if curr_open < stop_level:
						should_exit = True
					else:
						hwm = max(hwm, curr_high)
				else:
					stop_level = hwm + (prev_atr * params.atr_multiplier)
					df[i, "active_stop"] = stop_level
					if curr_open > stop_level:
						should_exit = True
					else:
						hwm = min(hwm, curr_low)
			elif (curr_state ==  2 and (curr_is_high_vol or curr_trend_down)) or \
				 (curr_state == -2 and (curr_is_high_vol or curr_trend_up)):
				should_exit = True

			if should_exit:
				if exit_lag > 0:
					exit_countdown = exit_lag - 1
				else:
					curr_state = 0

		if curr_state == 0 and exit_countdown == -1:
			hwm = curr_open
			if df[i, "mr_long_entry"]:
				curr_state = 1
			elif df[i, "mr_short_entry"]:
				curr_state = -1
			elif df[i, "tf_long_entry"]:
				curr_state = 2
			elif df[i, "tf_short_entry"]:
				curr_state = -2
			df[i, "entry_marker"] = curr_state

		df[i, "state"] = curr_state

	df = df.with_columns(
		final_position=pl.col("state").cast(pl.Float64).replace(
			{1: mr_exposure, -1: -mr_exposure, 2: tf_exposure, -2: -tf_exposure}
		)
	)
	pos_diff        = pl.col("final_position").diff().abs().fill_null(0.0).fill_nan(0.0)
	log_ret_bnh     = pl.col("open").log().diff().fill_null(0.0).alias("log_ret")
	open_ret        = pl.col("open").log().diff().shift(-1).fill_null(0.0)
	strat_ret_gross = (open_ret * pl.col("final_position")).alias("strat_ret_gross")
	strat_ret_net   = (strat_ret_gross - pos_diff * (commission + slippage)).alias("strat_ret_net")

	df = df.with_columns(
		log_ret_bnh,
		strat_ret_net,
		cum_ret       = log_ret_bnh.cum_sum(),
		strat_cum_ret = strat_ret_net.cum_sum(),
	)
	return df

In [ ]:
symbol     = "SPY"
start_date = datetime(year=2020, month=1, day=1)
end_date   = datetime.today()

TIMEFRAME: TimeFrame = TimeFrame.Hour  # type: ignore
BARS_PER_DAY = bars_per_day(TIMEFRAME)

strategy_params = StrategyHyperparams(
	timeframe = TIMEFRAME,
	# ── Vol regime parameters ────────────────────────────────────────────────
	band_std_dev         = 0.8,
	ema_span             = WEEK,
	zscore_window        = MONTH,
	vol_regime_threshold = 0.0,
	use_stress_flag      = True,
	# ── VWAP anchor control ──────────────────────────────────────────────────
	anchor_recompute = 1,
	# ── Trend following parameters ───────────────────────────────────────────
	trend_ma_window = 2 * WEEK,
	# ── Risk parameters ──────────────────────────────────────────────────────
	atr_period = WEEK,
	atr_multiplier = 3.0,
 )

# ── Fetch raw data (cached) ──────────────────────────────────────────────────
if "loaded" not in dir() or loaded != f"{symbol}{TIMEFRAME}":
	print(f"Fetching {symbol} data...")
	df_ohlcv, df_vix = fetch_data(symbol, start_date, end_date, TIMEFRAME)
	loaded = f"{symbol}{TIMEFRAME}"
	validate_vix_alignment(df_vix, df_ohlcv)


def run_strategy_bundle(df_ohlcv: pl.DataFrame, df_vix: pl.DataFrame, strategy_params: StrategyHyperparams) -> pl.DataFrame:
	df_combined = backtest(df_ohlcv, df_vix, strategy_params)
	df_trend    = backtest(df_ohlcv, df_vix, strategy_params, strategy_type="trend")
	df_mrev     = backtest(df_ohlcv, df_vix, strategy_params, strategy_type="mean_reversion")
	return df_combined.with_columns(
		trend_ret_net = df_trend["strat_ret_net"],
		trend_cum_ret = df_trend["strat_cum_ret"],
		mrev_ret_net  = df_mrev["strat_ret_net"],
		mrev_cum_ret  = df_mrev["strat_cum_ret"],
	)

df = run_strategy_bundle(df_ohlcv,  df_vix, strategy_params)

In [ ]:
# STRATEGY METRICS

def compute_drawdown(cum_log_ret: np.typing.ArrayLike):
	peak = np.maximum.accumulate(cum_log_ret)
	return np.exp(cum_log_ret - peak) - 1

def compute_strategy_stats(bars_per_day: float, log_ret: np.ndarray, bench_log_ret: np.ndarray|None = None, risk_free_rate=0.0) -> dict:
	total_ret = np.exp(np.sum(log_ret)) - 1

	annual_factor = bars_per_day * YEAR
	annual_ret = np.exp(np.mean(log_ret) * annual_factor) - 1
	annual_vol = np.std(log_ret) * np.sqrt(annual_factor)
 
	downside_rets = log_ret[log_ret < 0]
	downside_vol = np.std(downside_rets) * np.sqrt(annual_factor) if len(downside_rets) > 0 else 0
 
	sharpe = (annual_ret - risk_free_rate) / annual_vol if annual_vol != 0 else 0
	sortino = (annual_ret - risk_free_rate) / downside_vol if downside_vol != 0 else 0

	dd = compute_drawdown(np.cumsum(log_ret))
	max_dd = np.min(dd)
	avg_dd = np.mean(dd)

	if bench_log_ret is not None:
		cov_matrix = np.cov(log_ret, bench_log_ret)
		beta = cov_matrix[0, 1] / cov_matrix[1, 1]
		alpha_daily = np.mean(log_ret) - (beta * np.mean(bench_log_ret))
		ann_alpha = (1 + alpha_daily) ** annual_factor - 1
	else:
		beta = np.nan
		ann_alpha = np.nan

	return {
		"Total Return (%)": total_ret * 100,
		"Annualized Return (%)": annual_ret * 100,
		"Annualized Volatility (%)": annual_vol * 100,
		"Sharpe Ratio": sharpe,
		"Sortino Ratio": sortino,
		"Max Drawdown (%)": max_dd * 100,
		"Avg Drawdown (%)": avg_dd * 100,
		"Annualized Alpha (%)": ann_alpha * 100,
		"Beta (vs Asset)": beta,
	}

In [ ]:
# STRATEGY RESULTS
#
# Layout (7 panels):
#   1. Price — candlestick + vol bands + MAs + trailing stop + trade entries
#   2. VIX   — raw VIX, Kalman filtered level, one-step-ahead prediction
#   3. Regime Z-score — smoothed vol regime signal that gates MR vs TF mode
#   4. Term Slope — VIX3M − VIX (backwardation = stress)
#   5. Position
#   6. Cumulative log-returns (B&H, Hybrid, Trend, MR)
#   7. Drawdown

def plot_strategy(df: pl.DataFrame, bars_per_day: float, strategy_params: StrategyHyperparams):
	table_width = 0.62  # fraction of figure width reserved for the chart

	n      = len(df)
	x_axis = np.arange(n)

	# ── Pre-compute drawdowns ──────────────────────────────────────────────────
	dd_bench = compute_drawdown(df["cum_ret"].to_numpy())
	dd_strat = compute_drawdown(df["strat_cum_ret"].to_numpy())
	dd_trend = compute_drawdown(df["trend_cum_ret"].to_numpy())
	dd_mrev  = compute_drawdown(df["mrev_cum_ret"].to_numpy())

	# ── Full-period stats for the table ───────────────────────────────────────
	bench_stats = compute_strategy_stats(bars_per_day, df["log_ret"].to_numpy())
	strat_stats = compute_strategy_stats(bars_per_day, df["strat_ret_net"].to_numpy(), df["log_ret"].to_numpy())
	trend_stats = compute_strategy_stats(bars_per_day, df["trend_ret_net"].to_numpy(), df["log_ret"].to_numpy())
	mrev_stats  = compute_strategy_stats(bars_per_day, df["mrev_ret_net"].to_numpy(),  df["log_ret"].to_numpy())

	def fmt_stats(stats_list):
		return [[f"{v:.2f}" if not np.isnan(v) else "" for v in s.values()] for s in stats_list]

	# ── Colour palette ─────────────────────────────────────────────────────────
	C_BH     = "mediumpurple"
	C_STRAT  = "crimson"
	C_TREND  = "darkorange"
	C_MREV   = "seagreen"
	C_VIX    = "lightsteelblue"
	C_FILT   = "royalblue"
	C_REGIME = "purple"
	C_SLOPE_POS = "steelblue"
	C_SLOPE_NEG = "crimson"

	# ─────────────────────────────────────────────────────────────────────────
	# Build trace list
	# IMPORTANT: the last 8 traces must always be the 4 cum_ret + 4 drawdown
	# series — jump_to relies on this ordering to restyle the windowed view.
	# ─────────────────────────────────────────────────────────────────────────
	slope     = df["term_slope"].to_numpy()
	slope_ma  = df["term_slope_ma"].to_numpy()
	slope_colors = [C_SLOPE_NEG if v < 0 else C_SLOPE_POS for v in slope]

	data = [
		# ── Stats ────────────────────────────────────────────────────────────
		go.Table(
			header=dict(
				values=["Metric", "Buy & Hold", "Hybrid", "Trend", "Mean Rev."],
				align="left", font=dict(size=10),
			),
			cells=dict(
				values=[list(bench_stats.keys()), *fmt_stats([bench_stats, strat_stats, trend_stats, mrev_stats])],
				align="left", font=dict(size=10),
			),
			domain=dict(x=[table_width + 0.01, 1.0], y=[0, 1]),
		),

		# ── Candlestick ──────────────────────────────────────────────────────
		go.Candlestick(
			x=x_axis, open=df["open"], high=df["high"], low=df["low"], close=df["close"],
			name="Price", xaxis="x", yaxis="y",
			increasing_line_color="rgba(100,200,100,0.7)",
			decreasing_line_color="rgba(220,80,80,0.7)",
		),
		go.Scatter(
			x=x_axis, y=df["upper_band"], line_shape="hv",
			line=dict(color="rgba(100,160,220,0.6)", width=1, dash="dot"),
			name=f"+{strategy_params.band_std_dev}\u03c3 Band", legendgroup="Bands",
			xaxis="x", yaxis="y",
		),
		go.Scatter(
			x=x_axis, y=df["lower_band"], line_shape="hv",
			line=dict(color="rgba(100,160,220,0.6)", width=1, dash="dot"),
			fill="tonexty", fillcolor="rgba(100,160,220,0.07)",
			name=f"-{strategy_params.band_std_dev}\u03c3 Band", legendgroup="Bands",
			xaxis="x", yaxis="y",
		),
		
		go.Scatter(
			x=x_axis, y=df["trend_ma"],
			line=dict(color="mediumpurple", width=1.2, dash="dot"), opacity=0.9,
			name=f"Trend MA ({strategy_params.trend_ma_window_bars} bars)", legendgroup="MAs",
			xaxis="x", yaxis="y",
		),

		# ── Trailing stop ────────────────────────────────────────────────────
		go.Scatter(
			x=x_axis, y=df["active_stop"],
			line=dict(color="firebrick", width=1, dash="dash"), line_shape="hv",
			connectgaps=False, hoverinfo="skip",
			name=f"{strategy_params.atr_multiplier}\u00d7 ATR({strategy_params.atr_period_bars} bars) Stop",
			legendgroup="Trades", xaxis="x", yaxis="y",
		),

		# ── Trade entry markers (MR long/short, TF long/short) ───────────────
		go.Scatter(
			x=x_axis[df.select(pl.col("entry_marker") == 1).to_numpy().reshape(-1)],
			y=df.filter(pl.col("entry_marker") == 1)["low"].to_numpy() * 0.995,
			mode="markers", marker=dict(symbol="triangle-up", size=9, color=C_MREV),
			name="MR Long Entry", legendgroup="MR Trades", hoverinfo="skip",
			xaxis="x", yaxis="y",
		),
		go.Scatter(
			x=x_axis[df.select(pl.col("exit_marker") == 1).to_numpy().reshape(-1)],
			y=df.filter(pl.col("exit_marker") == 1)["high"].to_numpy() * 1.007,
			mode="markers", marker=dict(symbol="x-thin-open", size=5, color=C_MREV),
			name="MR Long Exit", legendgroup="MR Trades", hoverinfo="skip",
			xaxis="x", yaxis="y",
		),
		go.Scatter(
			x=x_axis[df.select(pl.col("entry_marker") == -1).to_numpy().reshape(-1)],
			y=df.filter(pl.col("entry_marker") == -1)["high"].to_numpy() * 1.007,
			mode="markers", marker=dict(symbol="triangle-down", size=9, color=C_MREV),
			name="MR Short Entry", legendgroup="MR Trades", hoverinfo="skip",
			xaxis="x", yaxis="y",
		),
		go.Scatter(
			x=x_axis[df.select(pl.col("exit_marker") == -1).to_numpy().reshape(-1)],
			y=df.filter(pl.col("exit_marker") == -1)["low"].to_numpy() * 0.995,
			mode="markers", marker=dict(symbol="x-thin-open", size=5, color=C_MREV),
			name="MR Short Exit", legendgroup="MR Trades", hoverinfo="skip",
			xaxis="x", yaxis="y",
		),
		# trend following
		go.Scatter(
			x=x_axis[df.select(pl.col("entry_marker") == 2).to_numpy().reshape(-1)],
			y=df.filter(pl.col("entry_marker") == 2)["low"].to_numpy() * 0.995,
			mode="markers", marker=dict(symbol="triangle-up", size=9, color=C_TREND),
			name="TF Long Entry", legendgroup="TF Trades", hoverinfo="skip",
			xaxis="x", yaxis="y",
		),
		go.Scatter(
			x=x_axis[df.select(pl.col("exit_marker") == 2).to_numpy().reshape(-1)],
			y=df.filter(pl.col("exit_marker") == 2)["high"].to_numpy() * 1.007,
			mode="markers", marker=dict(symbol="x-thin-open", size=5, color=C_TREND),
			name="TF Long Exit", legendgroup="TF Trades", hoverinfo="skip",
			xaxis="x", yaxis="y",
		),
		go.Scatter(
			x=x_axis[df.select(pl.col("entry_marker") == -2).to_numpy().reshape(-1)],
			y=df.filter(pl.col("entry_marker") == -2)["high"].to_numpy() * 1.007,
			mode="markers", marker=dict(symbol="triangle-down", size=9, color=C_TREND),
			name="TF Short Entry", legendgroup="TF Trades", hoverinfo="skip",
			xaxis="x", yaxis="y",
		),
		go.Scatter(
			x=x_axis[df.select(pl.col("exit_marker") == -2).to_numpy().reshape(-1)],
			y=df.filter(pl.col("exit_marker") == -2)["low"].to_numpy() * 0.995,
			mode="markers", marker=dict(symbol="x-thin-open", size=5, color=C_TREND),
			name="TF Short Exit", legendgroup="TF Trades", hoverinfo="skip",
			xaxis="x", yaxis="y",
		),
		# ── VIX panel ────────────────────────────────────────────────────────
		go.Scatter(
			x=x_axis, y=df["vix"],
			line=dict(color=C_VIX, width=1), opacity=0.7,
			name="VIX (raw)", xaxis="x", yaxis="y2",
		),
		go.Scatter(
			x=x_axis, y=df["vix_ema"],
			line=dict(color=C_FILT, width=2),
			name=f"VIX EMA ({strategy_params.ema_span})", xaxis="x", yaxis="y2",
		),

		# ── Vol regime z-score ───────────────────────────────────────────────
		go.Scatter(
			x=x_axis, y=df["vol_zscore"],
			fill="tozeroy",
			fillcolor="rgba(128,0,128,0.15)",
			line=dict(color=C_REGIME, width=1.5),
			name="Vol Z-Score", xaxis="x", yaxis="y3",
		),

		# ── Term structure slope ──────────────────────────────────────────────
		go.Bar(
			x=x_axis, y=slope,
			marker_color=slope_colors, opacity=0.5,
			name="Term Slope (VIX3M-VIX)", xaxis="x", yaxis="y4",
		),
		go.Scatter(
			x=x_axis, y=slope_ma,
			line=dict(color="black", width=1.5),
			name="21d Slope MA", xaxis="x", yaxis="y4",
		),

		# ── Position ──────────────────────────────────────────────────────────
		go.Scatter(
			x=x_axis, y=df["final_position"],
			fill="tozeroy", fillcolor="rgba(30,120,200,0.2)",
			line=dict(color="steelblue", width=1),
			name="Position", showlegend=False, xaxis="x", yaxis="y5",
		),

		# ── Cumulative returns ────────────────────────────────────────────────
		go.Scatter(x=x_axis, y=df["cum_ret"],       line=dict(color=C_BH,    width=1.5), name="Buy & Hold",    legendgroup="B&H",   xaxis="x", yaxis="y6"),
		go.Scatter(x=x_axis, y=df["strat_cum_ret"], line=dict(color=C_STRAT, width=2.0), name="Hybrid",        legendgroup="Hybrid", xaxis="x", yaxis="y6"),
		go.Scatter(x=x_axis, y=df["trend_cum_ret"], line=dict(color=C_TREND, width=1.5), name="Trend Only",    legendgroup="Trend",  xaxis="x", yaxis="y6"),
		go.Scatter(x=x_axis, y=df["mrev_cum_ret"],  line=dict(color=C_MREV,  width=1.5), name="Mean Rev.",     legendgroup="MRev",   xaxis="x", yaxis="y6"),

		# ── Drawdowns ─────────────────────────────────────────────────────────
		go.Scatter(x=x_axis, y=dd_bench, line=dict(color=C_BH,    width=1), name="B&H DD",    legendgroup="B&H",   showlegend=False, xaxis="x", yaxis="y7"),
		go.Scatter(x=x_axis, y=dd_strat, line=dict(color=C_STRAT, width=1), name="Hybrid DD", legendgroup="Hybrid", showlegend=False, xaxis="x", yaxis="y7"),
		go.Scatter(x=x_axis, y=dd_trend, line=dict(color=C_TREND, width=1), name="Trend DD",  legendgroup="Trend",  showlegend=False, xaxis="x", yaxis="y7"),
		go.Scatter(x=x_axis, y=dd_mrev,  line=dict(color=C_MREV,  width=1), name="MRev DD",   legendgroup="MRev",   showlegend=False, xaxis="x", yaxis="y7"),
	]

	# ── jump_to: windowed view + live stats update ─────────────────────────────
	def pad(lo, hi, p):
		r = hi - lo
		return lo - r * p, hi + r * p

	# last 8 traces = indices 18-25 (4 cum_ret + 4 drawdown)
	N_RETURN_TRACES = 8
	plot_indices = list(range(len(data) - N_RETURN_TRACES, len(data)))

	def jump_to(a: int, b: int, padx=0.05, pady=0.1):
		s0 = max(a, 0)
		s1 = min(b, n)
		s  = slice(s0, s1)

		def norm(col):
			arr = df[s, col].to_numpy()
			return arr - arr[0]

		bench_w = norm("cum_ret")
		strat_w = norm("strat_cum_ret")
		trend_w = norm("trend_cum_ret")
		mrev_w  = norm("mrev_cum_ret")

		dd_bench_w = compute_drawdown(bench_w)
		dd_strat_w = compute_drawdown(strat_w)
		dd_trend_w = compute_drawdown(trend_w)
		dd_mrev_w  = compute_drawdown(mrev_w)

		bw = compute_strategy_stats(bars_per_day, df[s, "log_ret"].to_numpy())
		sw = compute_strategy_stats(bars_per_day, df[s, "strat_ret_net"].to_numpy(), df[s, "log_ret"].to_numpy())
		tw = compute_strategy_stats(bars_per_day, df[s, "trend_ret_net"].to_numpy(), df[s, "log_ret"].to_numpy())
		mw = compute_strategy_stats(bars_per_day, df[s, "mrev_ret_net"].to_numpy(),  df[s, "log_ret"].to_numpy())

		table_vals = [list(bw.keys()), *fmt_stats([bw, sw, tw, mw])]
		xs = x_axis[s]

		restyle = {
			"cells.values": [table_vals] + [None] * N_RETURN_TRACES,
			"y": [None] + [bench_w, strat_w, trend_w, mrev_w,
						   dd_bench_w, dd_strat_w, dd_trend_w, dd_mrev_w],
			"x": [None] + [xs] * N_RETURN_TRACES,
		}

		# VIX range in window
		vix_lo = min(df[s, "vix"].min(), df[s, "vix_ema"].min()) # type: ignore
		vix_hi = max(df[s, "vix"].max(), df[s, "vix_ema"].max()) # type: ignore
		rz     = df[s, "vol_zscore"].to_numpy()
		sl     = df[s, "term_slope"].to_numpy()

		relayout = {
			"xaxis.range": pad(a, b, padx),
			"yaxis.range":  pad(
				min(df[s, "low"].min(), df[s, "lower_band"].min()), # type: ignore
				max(df[s, "high"].max(), df[s, "upper_band"].max()), # type: ignore
				pady,
			),
			"yaxis2.range": pad(vix_lo, vix_hi, pady),
			"yaxis3.range": pad(np.nanmin(rz), np.nanmax(rz), pady),
			"yaxis4.range": pad(np.nanmin(sl), np.nanmax(sl), pady),
			"yaxis6.range": pad(
				min(bench_w.min(), strat_w.min(), trend_w.min(), mrev_w.min()),
				max(bench_w.max(), strat_w.max(), trend_w.max(), mrev_w.max()),
				pady,
			),
			"yaxis7.range": [
				min(dd_bench_w.min(), dd_strat_w.min(), dd_trend_w.min(), dd_mrev_w.min()) * (1 + pady),
				0.005,
			],
		}
		return [restyle, relayout, [0] + plot_indices]

	# ── Row heights + y-domain layout ─────────────────────────────────────────
	spacing = 0.012
	rh_raw  = np.array([3.0, 1.2, 0.8, 0.8, 0.6, 1.8, 0.8])   # price, vix, regime, slope, pos, returns, dd
	rh      = rh_raw / rh_raw.sum()
	rh_rev  = rh[::-1]
	ydomains = [
		(np.sum(rh_rev[:i]) + spacing * (i > 0),
		 np.sum(rh_rev[:i+1]) - spacing * (i < len(rh_rev)-1))
		for i in range(len(rh_rev))
	][::-1]

	layout = dict(
		hoversubplots="axis",
		hovermode="x",
		legend=dict(orientation="h", yanchor="top", y=-0.02, xanchor="left", x=0, font=dict(size=10)),
		xaxis=dict(
			domain=[0, table_width],
			showticklabels=False, showgrid=False,
			showspikes=True, spikemode="across", spikethickness=1,
			tickvals=x_axis,
			ticktext=df["timestamp"].dt.strftime("%a %d %b %Y %H:%M %Z").to_list(),
			rangeslider=dict(visible=False),
		),
		yaxis =dict(domain=ydomains[0], title="Price",         automargin=True),
		yaxis2=dict(domain=ydomains[1], title="VIX",           automargin=True),
		yaxis3=dict(domain=ydomains[2], title="Vol Z",      automargin=True, zeroline=True, zerolinecolor="black", zerolinewidth=1),
		yaxis4=dict(domain=ydomains[3], title="Term Slope",    automargin=True, zeroline=True, zerolinecolor="black", zerolinewidth=1),
		yaxis5=dict(domain=ydomains[4], title="Position",      automargin=True, zeroline=True, zerolinecolor="gray"),
		yaxis6=dict(domain=ydomains[5], title="Cum. Log Ret.", automargin=True),
		yaxis7=dict(domain=ydomains[6], title="Drawdown",      automargin=True),
		updatemenus=[dict(
			type="buttons", direction="right",
			x=table_width, y=1.045,
			buttons=[
				dict(label="All",  method="update", args=jump_to(0, n)),
				dict(label="252D", method="update", args=jump_to(n - ceil(bars_per_day * YEAR),        n)),
				dict(label="63D",  method="update", args=jump_to(n - ceil(bars_per_day * MONTH * 3),  n)),
				dict(label="21D",  method="update", args=jump_to(n - ceil(bars_per_day * MONTH),      n)),
				dict(label="10D",  method="update", args=jump_to(n - ceil(bars_per_day * WEEK * 2),   n)),
				dict(label="5D",   method="update", args=jump_to(n - ceil(bars_per_day * WEEK),       n)),
			],
		)],
		barmode="overlay",
	)

	fig = go.Figure(data=data, layout=layout)

	is_high_vol = df["is_high_vol"].fill_null(False).to_numpy()
	regime_start = None
	regime_fill = "rgba(128, 0, 128, 0.06)"

	for i, flag in enumerate(is_high_vol):
		if flag and regime_start is None:
			regime_start = i
		elif not flag and regime_start is not None:
			fig.add_vrect(
				x0=regime_start - 0.5,
				x1=i - 0.5,
				xref="x",
				yref="y",
				y0=0.0,
				y1=1.0,
				fillcolor=regime_fill,
				line_width=0,
				layer="below",
			)
			regime_start = None

	if regime_start is not None:
		fig.add_vrect(
			x0=regime_start - 0.5,
			x1=n - 0.5,
			xref="x",
			yref="y",
			y0=0.0,
			y1=1.0,
			fillcolor=regime_fill,
			line_width=0,
			layer="below",
		)

	return fig


In [ ]:
# PLOT STRATEGY

fig = plot_strategy(df, BARS_PER_DAY, strategy_params) \
	.update_layout(
		title=f"{symbol}: Hybrid Strategy "
			  f"({df['timestamp'].min().strftime('%b %Y')} "
			  f"→ {df['timestamp'].max().strftime('%b %Y')})",
		width=1000, height=1000,
	)
fig.write_image(f"fig/{symbol}_strategy.jpg")
fig.show()

In [ ]:
# MONTE CARLO SIMULATION

def stationary_bootstrap_monte_carlo(log_ret: np.ndarray, avg_block_size: int, n_simulations=1000, random_state=np.random.RandomState(42)):
	n_steps = len(log_ret)
	simulated_paths: list[np.ndarray] = [None] * n_simulations # type: ignore
	simulated_dd: list[np.ndarray] = [None] * n_simulations # type: ignore
	
	# p is the probability of starting a new block
	p = 1.0 / avg_block_size
	
	for i in range(n_simulations):
		indices = np.zeros(n_steps, dtype=int)
		# Pick the very first day randomly
		current_idx = random_state.randint(0, n_steps)
		indices[0] = current_idx
		
		for t in range(1, n_steps):
			if random_state.rand() < p:
				# 'Jump' to a new random starting point (New Block)
				current_idx = random_state.randint(0, n_steps)
			else:
				# Stay in the current block (Next consecutive day)
				current_idx = (current_idx + 1) % n_steps
			
			indices[t] = current_idx
			
		simulated_returns = log_ret[indices]
		log_ret_path = np.cumsum(simulated_returns)
		simulated_paths[i] = log_ret_path
		simulated_dd[i] = compute_drawdown(log_ret_path)
		
	return simulated_paths, simulated_dd

def plot_monte_carlo(mc_paths: list[np.ndarray], mc_dd: list[np.ndarray], cum_log_ret: np.ndarray, show_paths=100):
	x_axis = np.arange(len(cum_log_ret))
 
	final_equity = [path[-1] for path in mc_paths]
	max_dd = [np.min(dd) for dd in mc_dd]
 
	path_p5 = np.percentile(mc_paths, 5, axis=0)
	path_p50 = np.percentile(mc_paths, 50, axis=0)
	path_p95 = np.percentile(mc_paths, 95, axis=0)
 
	dd_p5 = np.percentile(mc_dd, 5, axis=0)
	dd_p50 = np.percentile(mc_dd, 50, axis=0)
	dd_p95 = np.percentile(mc_dd, 95, axis=0)
	
	max_dd_p5 = np.percentile(max_dd, 5)
	max_dd_p50 = np.percentile(max_dd, 50)
	max_dd_p95 = np.percentile(max_dd, 95)
 
	dd_r = compute_drawdown(cum_log_ret)
 
	fig = make_subplots(rows=2, cols=2, shared_xaxes="columns", shared_yaxes="rows", column_widths=[4, 1], subplot_titles=["Simulation Cum. Log Returns", "Final Return Distribution", "Simulation Drawdowns", "Max DD Distribution"], horizontal_spacing=0.02, vertical_spacing=0.05)
	for i in range(show_paths):
		c = px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
		fig.add_trace(
			go.Scatter(
				x=x_axis,
				y=mc_paths[i],
				line_color=c,
				opacity=0.2,
				hoverinfo="skip",
				showlegend=False
			), row=1, col=1)
		fig.add_trace(
			go.Scatter(
				x=x_axis,
				y=mc_dd[i],
				line_color=c,
				opacity=0.2,
				hoverinfo="skip",
				showlegend=False
			), row=2, col=1)
  
		
	fig.add_traces([
		go.Scatter(x=x_axis, y=path_p95, line_color="gray", name="95th Percentile", legendgroup="95"),
		go.Scatter(x=x_axis, y=path_p50, line_color="blue", name="50th Percentile", legendgroup="50"),
		go.Scatter(x=x_axis, y=path_p5, line_color="gray", name="5th Percentile", legendgroup="5"),
		go.Scatter(x=x_axis, y=cum_log_ret, line_color="red", name="Realized Strategy Path", legendgroup="R"),
	], rows=1, cols=1)
 
	fig.add_traces([
		go.Scatter(x=x_axis, y=dd_p95, line_color="gray", name="Drawdown 95th Percentile", legendgroup="95"),
		go.Scatter(x=x_axis, y=dd_p50, line_color="blue", name="Drawdown 50th Percentile", legendgroup="50"),
		go.Scatter(x=x_axis, y=dd_p5, line_color="gray", name="Drawdown 5th Percentile", legendgroup="5"),
		go.Scatter(x=x_axis, y=dd_r, line_color="red", fill="tozeroy", name="Realized Drawdown", legendgroup="R"),
	], rows=2, cols=1)

	fig.add_trace(go.Histogram(y=final_equity, nbinsx=50, histnorm="probability", marker_color="purple", showlegend=False), row=1, col=2)
	fig.add_hline(y=path_p95[-1], line_color="gray", row=1, col=2)
	fig.add_hline(y=path_p50[-1], line_color="blue", line_dash="dot", row=1, col=2)
	fig.add_hline(y=path_p5[-1], line_color="gray", row=1, col=2)
	fig.add_hline(y=cum_log_ret[-1], line_color="red", row=1, col=2)

	fig.add_trace(go.Histogram(y=max_dd, nbinsx=50, histnorm="probability", marker_color="purple", showlegend=False), row=2, col=2)
	fig.add_hline(y=max_dd_p95, line_color="gray", row=2, col=2)
	fig.add_hline(y=max_dd_p50, line_color="blue", line_dash="dot", row=2, col=2)
	fig.add_hline(y=max_dd_p5, line_color="gray", row=2, col=2)
	fig.add_hline(y=np.min(dd_r), line_color="red", row=2, col=2)
 
	fig.update_xaxes(showgrid=False, showticklabels=False)

	return fig

In [ ]:
# PLOT MONTE CARLO

AVG_BLOCK_SIZE_D = ceil(3 * WEEK * BARS_PER_DAY)
mc_paths, mc_dd = stationary_bootstrap_monte_carlo(df["strat_ret_net"].to_numpy(), AVG_BLOCK_SIZE_D)

fig = plot_monte_carlo(mc_paths, mc_dd, df["strat_cum_ret"].to_numpy(), show_paths=100) \
	.update_layout(title=f"{symbol}: Monte Carlo Simulation of Strategy Returns", width=1000, height=600)
fig.write_image(f"fig/{symbol}_monte_carlo.jpg")
fig.show()


In [ ]:
# TIMEFRAME SENSITIVITY

def run_timeframe_sensitivity(timeframes: list[TimeFrame], symbol: str, start_date: datetime, end_date: datetime, strategy_params: StrategyHyperparams):
	df_ohlcv, df_vix = fetch_data(symbol, start_date, end_date, TimeFrame.Minute) # type: ignore
	
	results = []
	paths: list[tuple[TimeFrame, pl.DataFrame]] = []
 
	for timeframe in (pbar := tqdm(timeframes)):
		try:
			pbar.set_description(f"{timeframe}")
			pbar.set_postfix_str("resampling data")
			resample_unit = {
				TimeFrameUnit.Day: "d",
				TimeFrameUnit.Hour: "h",
				TimeFrameUnit.Minute: "m",
			}[timeframe.unit]
			df_resample = (
				df_ohlcv
				.group_by_dynamic("timestamp", every=f"{timeframe.amount}{resample_unit}")
				.agg(pl.col("open").first(),pl.col("high").max(),pl.col("low").min(),pl.col("close").last(),pl.col("volume").sum())
				.sort("timestamp")
			)
			pbar.set_postfix_str("running backtest")
			df = backtest(df_resample, df_vix, strategy_params._replace(timeframe=timeframe))
			pbar.set_postfix_str("computing stats")
			stats = compute_strategy_stats(bars_per_day(timeframe), df["strat_ret_net"].to_numpy(), df["log_ret"].to_numpy())
			paths.append((timeframe, df.select("date", "strat_cum_ret")))
			results.append({"timeframe": timeframe, **stats})
			pbar.set_postfix_str(f"sharpe={stats['Sharpe Ratio']:.2f}, annual_ret={stats['Annualized Return (%)']:.2f}%")
			
		except Exception as e:
			print(f"Error in iteration {timeframe}: {e}")
	return pd.DataFrame(results), paths

def plot_timeframe_sensitivity(timeframe_results: pd.DataFrame, timeframe_paths: list[tuple[TimeFrame, pl.DataFrame]]):
	fig = make_subplots(rows=4, cols=1, subplot_titles=["Paths", "Sharpe Ratio", "Annualized Alpha (%)", "Beta (vs Asset)"], row_heights=[1.5, 1, 1, 1], vertical_spacing=0.075)
	
	fig.add_traces([
		go.Scatter(
			x=path["date"],
			y=path["strat_cum_ret"],
			line_color=px.colors.sequential.Plasma[i+1],
			opacity=0.7,
			mode="lines",
			name=str(timeframe),
			hovertemplate=f"Timeframe: {timeframe}<br>Total Return: {np.exp(path[-1,'strat_cum_ret'])-1:.2%}",
		) for i, (timeframe, path) in enumerate(timeframe_paths)
	])
 
	for row, metric in enumerate(["Sharpe Ratio", "Annualized Alpha (%)", "Beta (vs Asset)"], start=2):
		fig.add_trace(go.Scatter(
			x=timeframe_results["timeframe"].astype(str),
			y=timeframe_results[metric],
			name=metric,
			marker_color=px.colors.qualitative.Plotly[row],
			mode="lines+markers",
			showlegend=False,
		), row=row, col=1)
  
	return fig


In [ ]:
# PLOT TIMEFRAME SENSITIVITY

# type: ignore
timeframes: list[TimeFrame] = [
	TimeFrame(5, TimeFrameUnit.Minute),
	TimeFrame(15, TimeFrameUnit.Minute),
	TimeFrame(30, TimeFrameUnit.Minute),
	TimeFrame.Hour,
	TimeFrame(2, TimeFrameUnit.Hour),
	TimeFrame(4, TimeFrameUnit.Hour),
]

OVERWRITE = False
if not os.path.exists(path := f"tmp/{symbol}_timeframe_results.pkl") or OVERWRITE:
	timeframe_results, timeframe_paths = run_timeframe_sensitivity(
		timeframes, symbol, start_date, end_date, strategy_params
	)
	with open(path, "wb") as f:
		pickle.dump((timeframe_results, timeframe_paths), f)
else:
	with open(path, "rb") as f:
		timeframe_results, timeframe_paths = pickle.load(f)

fig = plot_timeframe_sensitivity(timeframe_results, timeframe_paths) \
	.update_layout(title=f"{symbol}: Timeframe Sensitivity Analysis", width=1000, height=700)
fig.write_image(f"fig/{symbol}_timeframe_sensitivity.jpg")
fig.show()


In [ ]:
# LAG SENSITIVITY

from statsmodels.regression.linear_model import OLS

def run_lag_sensitivity(min_lag: int, max_lag: int, df_ohlcv: pl.DataFrame, df_vix: pl.DataFrame, strategy_params: StrategyHyperparams, mode="both"):
	results = []
	paths = []
 
	for lag_period in (pbar := tqdm(range(min_lag, max_lag + 1))):
		try:
			if mode == "entry":
				entry_lag, exit_lag = lag_period, 0
			elif mode == "exit":
				entry_lag, exit_lag = 1, lag_period
			elif mode == "both":
				entry_lag, exit_lag = lag_period, lag_period
			else:
				raise ValueError("mode must be 'entry', 'exit', or 'both'")

			df = backtest(df_ohlcv, df_vix, strategy_params, entry_lag=entry_lag, exit_lag=exit_lag)
			stats = compute_strategy_stats(strategy_params.bpd, df["strat_ret_net"].to_numpy(), df["log_ret"].to_numpy())
			paths.append((lag_period, df["strat_cum_ret"]))
			results.append({"lag_period": lag_period, **stats})
			pbar.set_postfix_str(f"sharpe={stats['Sharpe Ratio']:.2f}, annual_ret={stats['Annualized Return (%)']:.2f}%")
			
		except Exception as e:
			print(f"Error in iteration {lag_period}: {e}")

	return pd.DataFrame(results), paths

def plot_lag_sensitivity(lag_outputs: list[tuple[pd.DataFrame, list[tuple[int, pl.Series]]]], cmin: int, cmax: int, column_titles: list[str] | None = None):
	fig = make_subplots(rows=4, cols=len(lag_outputs), shared_yaxes=True, row_titles=["Paths", "Sharpe Ratio", "Alpha", "Beta"], column_titles=column_titles, row_heights=[1.5, 1, 1, 1], horizontal_spacing=0.02, vertical_spacing=0.04)
	
	for col, (lag_results, lag_paths) in enumerate(lag_outputs, start=1):
		fig.add_traces([
			go.Scatter(
				x=np.arange(len(path)),
				y=path,
				line_color=px.colors.sample_colorscale("Plasma", [lag / lag_results["lag_period"].max()])[0],
				opacity=0.5,
				mode="lines",
				name=f"Lag {lag}",
				hovertemplate=f"Lag Period: {lag}<br>Total Return: {np.exp(path[-1])-1:.2%}",
				showlegend=False
			) for lag, path in lag_paths
		], rows=1, cols=col)

		for row, metric in enumerate(["Sharpe Ratio", "Annualized Alpha (%)", "Beta (vs Asset)"], start=2):
			
			X = lag_results["lag_period"].to_numpy().reshape(-1, 1)
			y = lag_results[metric].to_numpy()
			model = OLS(y, np.hstack([np.ones_like(X), X])).fit()
			y_pred = model.predict(np.hstack([np.ones_like(X), X]))
		
			fig.add_traces([
				go.Scatter(
					x=lag_results["lag_period"],
					y=lag_results[metric],
					name=metric,
					marker_color=px.colors.qualitative.Plotly[row],
					mode="markers",
					showlegend=False,
				),
				go.Scatter(
					x=lag_results["lag_period"],
					y=y_pred,
					name="OLS Fit",
					hovertemplate=f"{model.params[0]:.2f} {'+' if model.params[1] >= 0 else '-'} {abs(model.params[1]):.4f} * Lag Period",
					line_color=px.colors.qualitative.Plotly[row],
					mode="lines",
					showlegend=False,
				),
		], rows=row, cols=col)

	fig.add_trace(
		go.Scatter(
			x=[None], y=[None],
			mode='markers',
			marker=dict(
				colorscale="Plasma",
				showscale=True,
				cmin=cmin,
				cmax=cmax,
				colorbar=dict(title="Lag Period", y=0.5, len=0.8),
			),
			hoverinfo='none',
			showlegend=False
		)
	)
	return fig


In [ ]:
# PLOT LAG SENSITIVITY

min_lag, max_lag = 0, ceil(WEEK * BARS_PER_DAY)
if "entry_lag_out" not in dir():
	entry_lag_out = run_lag_sensitivity(min_lag, max_lag, df_ohlcv, df_vix, strategy_params, mode="entry")
	exit_lag_out  = run_lag_sensitivity(min_lag, max_lag, df_ohlcv, df_vix, strategy_params, mode="exit")

fig = plot_lag_sensitivity([entry_lag_out, exit_lag_out], cmin=min_lag, cmax=max_lag, column_titles=["Variable Entry Lag (Exit Lag = 0)", "Variable Exit Lag (Entry Lag = 1)"]) \
	.update_layout(title=f"{symbol}: Lag Sensitivity Analysis", width=1200, height=700)
fig.write_image(f"fig/{symbol}_lag_sensitivity.jpg")
fig.show()


In [ ]:
# PARAMETER SENSITIVITY ANALYSIS

from itertools import combinations

def run_parameter_sensitivity(
	df_ohlcv: pl.DataFrame,
	df_vix: pl.DataFrame,
	bars_per_day: float,
	strategy_params: StrategyHyperparams,
	iterations=500,
	random_state=np.random.RandomState(42),
):
	results = []
	paths = []
 
	for i in (pbar := tqdm(range(iterations))):

		trial_params = StrategyHyperparams(
			timeframe = strategy_params.timeframe,
			# ── Vol regime parameters ────────────────────────────────────────────────
			band_std_dev         = round(float(random_state.uniform(0.5, 1.5)), 2),
			ema_span             = int(random_state.randint(3, 15)),
			zscore_window        = int(random_state.randint(21, 126)),
			vol_regime_threshold = round(float(random_state.uniform(-0.5, 1.5)), 2),
			use_stress_flag 	 = True,
			# ── VWAP anchor control ──────────────────────────────────────────────────
			anchor_recompute = int(random_state.randint(5, 3 * MONTH)),
			# ── Trend following parameters ───────────────────────────────────────────
			trend_ma_window = int(random_state.randint(5, 3 * MONTH)),
			# ── Risk parameters ──────────────────────────────────────────────────────
			atr_period 	   = int(random_state.randint(3, MONTH)),
			atr_multiplier = round(float(random_state.uniform(1.5, 4.0)), 1),
		)


		try:
			df = backtest(df_ohlcv, df_vix, trial_params)
			stats = compute_strategy_stats(bars_per_day, df["strat_ret_net"].to_numpy(), df["log_ret"].to_numpy())
			paths.append((trial_params, df["strat_cum_ret"]))
			results.append({**trial_params._asdict(), **stats})
			pbar.set_postfix_str(f"sharpe={stats['Sharpe Ratio']:.2f}, annual_ret={stats['Annualized Return (%)']:.2f}%")
			
		except Exception as e:
			print(f"Error in iteration {i}: {e}")

	return pd.DataFrame(results), paths


def plot_parameter_sensitivity(sensitivity_results: pd.DataFrame, metric="Sharpe Ratio"):
	mr_params = {"band_std_dev", "ema_span", "zscore_window", "vol_regime_threshold", "atr_period", "atr_multiplier"}
	tf_params = {"ema_span", "zscore_window", "vol_regime_threshold", "trend_ma_window"}
	params = mr_params.union(tf_params)
	param_pairs = [
		pair for pair in combinations(params, 2)
		if (set(pair).issubset(mr_params) or set(pair).issubset(tf_params))
	]
	for param in params:
		sensitivity_results[f"{param}_bin"] = pd.cut(sensitivity_results[param], bins=6)

	fig = make_subplots(rows=len(param_pairs) // 3 + (len(param_pairs) % 3 > 0), cols=3, subplot_titles=[f"x={x} vs. y={y}" for x,y in param_pairs], horizontal_spacing=0.1, vertical_spacing=0.06)

	for i, (param_x, param_y) in enumerate(param_pairs):
		row = (i // 3) + 1
		col = (i % 3) + 1
		
		hm = (
			sensitivity_results
			.pivot_table(index=f"{param_y}_bin", columns=f"{param_x}_bin", values=metric, aggfunc="mean", observed=True)
			.fillna(0)
		)
	
		
		fig.add_trace(
			go.Heatmap(
				z=hm.values,
				x=[f"({i.left:.2f}, {i.right:.2f}]" for i in hm.columns],
				y=[f"({i.left:.2f}, {i.right:.2f}]" for i in hm.index],
				texttemplate="%{z:.2f}",
				textfont_size=10,
				hovertemplate=
					f"{param_x}: %{{x}}<br>"
					f"{param_y}: %{{y}}<br>"
					f"{metric}: %{{z:.2f}}<extra></extra>",
				coloraxis="coloraxis"
			),
			row=row, col=col
		)
		
	fig.update_xaxes(tickfont_size=9)
	fig.update_yaxes(tickfont_size=9)
	fig.update_annotations(font_size=12)
	fig.update_layout(coloraxis=dict(colorbar=dict(title=metric, y=0.5, x=1.05, len=0.8), colorscale="RdBu", cmid=0))
 
	return fig


In [ ]:
# PLOT PARAMETER SENSITIVITY

OVERWRITE = False
if not os.path.exists(path := f"tmp/{symbol}_sensitivity_results.pkl") or OVERWRITE:
	sensitivity_results, paths = run_parameter_sensitivity(df_ohlcv, df_vix, BARS_PER_DAY, strategy_params)
	with open(path, "wb") as f:
		pickle.dump((sensitivity_results, paths), f)
else:
	with open(path, "rb") as f:
		sensitivity_results, paths = pickle.load(f)
  
fig = plot_parameter_sensitivity(sensitivity_results, metric="Annualized Alpha (%)") \
	.update_layout(title=f"{symbol}: Parameter Sensitivity Analysis", width=1200, height=1200)
fig.write_image(f"fig/{symbol}_parameter_sensitivity.jpg")
fig.show()
